# 🌪️ Tweet Bencana (*Disaster Tweets*) — Baseline DistilBERT (PyTorch)

> **Notebook 1 dari 2.** Ini adalah **baseline**: penyetelan halus (*fine-tuning*) DistilBERT dengan PyTorch + Hugging Face, lengkap dengan penjelasan baris per baris.
> Notebook lanjutan untuk meningkatkan skor ada di `disaster-tweets-roberta-cv-pytorch-id.ipynb`.

| Item | Keterangan |
|---|---|
| **Tugas** | Klasifikasi biner: apakah sebuah tweet membicarakan bencana nyata (`1`) atau bukan (`0`) |
| **Data** | ±7.6 ribu tweet berlabel (`train.csv`) dan ±3.2 ribu tweet tanpa label (`test.csv`) |
| **Metrik kompetisi** | **F1-score** pada kelas `1` |
| **Model** | DistilBERT (versi ringkas BERT) + kepala klasifikasi (*classification head*) 2 kelas |
| **Framework** | PyTorch + 🤗 Transformers (**tanpa** Keras / TensorFlow) |
| **Skor Kaggle (*public leaderboard*)** | **0.83389** ← angka yang harus dikalahkan oleh notebook 2 |

---

## 📑 Daftar isi
1. [Persiapan: import & konfigurasi](#1.-Persiapan:-import-&-konfigurasi)
2. [Memuat data](#2.-Memuat-data)
3. [Eksplorasi data](#3.-Eksplorasi-data)
4. [Membagi data latih / validasi](#4.-Membagi-data-latih-/-validasi)
5. [Memuat model & tokenizer](#5.-Memuat-model-&-tokenizer)
6. [DataLoader](#6.-DataLoader)
7. [Pelatihan model](#7.-Pelatihan-model)
8. [Evaluasi](#8.-Evaluasi)
9. [Membuat file submission](#9.-Membuat-file-submission)
10. [Rangkuman & keterbatasan baseline](#10.-Rangkuman-&-keterbatasan-baseline)

---

## 🗺️ Alur keseluruhan

```text
train.csv ──► bersihkan NaN ──► bagi 80/20 ──► tokenisasi ──► DistilBERT ──► logits (2 angka)
                                                                   ▲               │
                                              perbarui bobot ◄── backward ◄── loss (cross-entropy)
                                                                                   │
                                                  validasi (F1) ──► bobot terbaik ──► prediksi data uji ──► submission.csv
```

## 🔎 Cara membaca notebook ini
- Setiap **sel kode** diawali penjelasan konsep di markdown, lalu **setiap baris kode diberi komentar `#`** di sebelah kanan / di atasnya.
- Kotak bertanda `> 💡` = tips, `> ⚠️` = hal yang sering bikin error, `> 🧠` = intuisi di balik konsep.
- Bagian `▸ Detail` bisa dibuka-tutup (klik) untuk penjelasan tambahan.

# 1. Persiapan: import & konfigurasi

## 1.1 Import library

| Library | Dipakai untuk |
|---|---|
| `torch` | Tensor, autograd (turunan otomatis), optimizer, GPU |
| `transformers` | Model DistilBERT pra-latih + tokenizer-nya |
| `pandas` / `numpy` | Membaca CSV dan operasi array |
| `scikit-learn` | Pembagian data dan metrik (F1, matriks konfusi) |
| `matplotlib` | Menggambar matriks konfusi (*confusion matrix*) |
| `tqdm` | Bilah kemajuan (*bilah kemajuan*) saat pelatihan |

> 💡 **Kenapa `transformers`?** Library ini menyediakan arsitektur DistilBERT yang sudah ditulis dalam PyTorch, sehingga kita tidak perlu menulis ulang Transformer dari nol.

In [ ]:
# !pip install -q torch transformers scikit-learn pandas matplotlib tqdm   # (opsional) jalankan sekali kalau library belum terpasang

import os                      # fungsi sistem file: cek path ada/tidak, gabung path, list isi folder
import copy                    # copy.deepcopy: menyalin bobot model terbaik supaya tidak ikut berubah saat training lanjut
import random                  # generator acak bawaan Python (untuk reproducibility)
import contextlib              # contextlib.nullcontext: "context manager kosong" saat mixed precision dimatikan

import numpy as np             # operasi array numerik (argmax, seed, dsb.)
import pandas as pd            # membaca CSV & mengolah tabel (DataFrame)
import torch                   # library inti deep learning: tensor, autograd, GPU
from torch.utils.data import Dataset, DataLoader   # Dataset = cara ambil 1 contoh; DataLoader = menyusun contoh jadi batch
from transformers import (
    AutoTokenizer,                         # memuat tokenizer yang cocok dengan model secara otomatis
    AutoModelForSequenceClassification,    # memuat model + head klasifikasi sesuai arsitektur model
    get_linear_schedule_with_warmup,       # penjadwal learning rate: naik pelan-pelan (warmup) lalu turun linear
)
from sklearn.model_selection import train_test_split                        # membagi data jadi train & validation
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score   # metrik evaluasi
import matplotlib.pyplot as plt    # membuat grafik
from tqdm.auto import tqdm         # bilah kemajuan (otomatis cocok untuk notebook maupun terminal)

import transformers                # di-import terpisah hanya untuk mencetak versinya
print("Versi Torch:", torch.__version__)               # cetak versi PyTorch
print("Versi Transformers:", transformers.__version__) # cetak versi Transformers

## 1.2 Hyperparameter & perangkat (*device*)

**Hyperparameter** = pengaturan yang *kita* tentukan sebelum training (bukan dipelajari model).

| Nama | Nilai | Arti | Kalau dinaikkan |
|---|---|---|---|
| `SEED` | 42 | Titik awal angka acak | — (hanya agar hasil bisa diulang) |
| `MAX_LEN` | 160 | Batas jumlah token per tweet | Lebih banyak teks tertampung, tapi lebih lambat |
| `BATCH_SIZE` | 32 | Jumlah tweet yang diproses sekali jalan | Lebih cepat per epoch, butuh memori lebih besar |
| `EPOCHS` | 2 | Berapa kali seluruh data latih dilihat | Risiko *overfitting* naik |
| `LR` | 3e-5 | *Learning rate*: besar langkah update bobot | Terlalu besar → pelatihan tidak stabil |
| `WEIGHT_DECAY` | 0.01 | Regularisasi L2 agar bobot tidak membesar | Model lebih "tertahan", mengurangi overfitting |
| `WARMUP_RATIO` | 0.1 | 10% langkah pertama LR naik dari 0 | Awal pelatihan lebih stabil |
| `VAL_SPLIT` | 0.2 | 20% data disisihkan untuk validasi | Validasi lebih andal, data latih berkurang |

> 🧠 **Presisi campuran (*mixed precision*, AMP)**: di GPU NVIDIA, sebagian hitungan memakai float16 (setengah presisi) agar lebih cepat dan hemat memori. Di CPU / Apple GPU kita matikan.

In [ ]:
# ===== Konfigurasi =====
SEED = 42                                   # angka awal generator acak → hasil bisa direproduksi

# Folder model lokal (format Hugging Face: config.json + model.safetensors / pytorch_model.bin + vocab.txt)
MODEL_DIR = "../pretrained/distilbert_en"   # lokasi model pra-latih di komputermu
# Dipakai hanya jika MODEL_DIR tidak valid (butuh internet)
FALLBACK_MODEL = "distilbert-base-uncased"  # nama model di Hugging Face Hub sebagai cadangan

MAX_LEN = 160                               # tweet maksimal 280 karakter → 160 token sudah lebih dari cukup
BATCH_SIZE = 32                             # jumlah tweet per batch
EPOCHS = 2                                  # berapa putaran penuh atas data train
LR = 3e-5                                   # learning rate puncak (setelah warmup)
WEIGHT_DECAY = 0.01                         # kekuatan regularisasi L2
WARMUP_RATIO = 0.1                          # porsi langkah awal untuk warmup
VAL_SPLIT = 0.2                             # porsi data untuk validasi


def set_seed(seed):                         # fungsi pengunci semua sumber keacakan
    random.seed(seed)                       # keacakan Python
    np.random.seed(seed)                    # keacakan NumPy (dipakai antara lain oleh sklearn)
    torch.manual_seed(seed)                 # keacakan PyTorch di CPU (inisialisasi head, dropout, shuffle)
    torch.cuda.manual_seed_all(seed)        # keacakan PyTorch di semua GPU (aman dipanggil walau tak ada GPU)

set_seed(SEED)                              # panggil sekali di awal

if torch.cuda.is_available():               # ada GPU NVIDIA?
    device = torch.device("cuda")           #   → pakai GPU NVIDIA
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():   # Mac Apple Silicon?
    device = torch.device("mps")            #   → pakai GPU Apple
else:                                       # selain itu
    device = torch.device("cpu")            #   → pakai CPU (lambat tapi tetap jalan)

use_amp = device.type == "cuda"             # mixed precision (fp16) hanya diaktifkan di GPU NVIDIA
print("Perangkat:", device, "| AMP:", use_amp) # tampilkan perangkat yang dipakai

# 2. Memuat data

Kolom pada data:

| Kolom | Isi | Catatan |
|---|---|---|
| `id` | ID unik tweet | Tidak dipakai sebagai fitur |
| `keyword` | Kata kunci dari tweet | Boleh kosong (`NaN`) |
| `location` | Lokasi pengirim | Boleh kosong, sangat berisik |
| `text` | **Isi tweet** | Fitur utama kita |
| `target` | `1` = bencana nyata, `0` = bukan | Hanya ada di `train.csv` |

> ⚠️ **Perhatian:** `find_file` mencari di folder kerja dulu, lalu di `/kaggle/input/nlp-getting-started`. Jadi notebook yang sama jalan di laptop **dan** di Kaggle tanpa diubah.

In [ ]:
KAGGLE_DIR = "/kaggle/input/nlp-getting-started"      # lokasi data kompetisi jika dijalankan di Kaggle

def find_file(filename):                               # fungsi: cari file di beberapa folder kandidat
    """Cari file di folder kerja, lalu di folder input Kaggle."""
    for folder in [".", KAGGLE_DIR]:                   # urutan pencarian: folder saat ini dulu, baru folder Kaggle
        path = os.path.join(folder, filename)          # gabungkan folder + nama file → path lengkap
        if os.path.exists(path):                       # kalau file-nya ada
            return path                                #   kembalikan path itu
    raise FileNotFoundError(f"{filename} tidak ditemukan di '.' maupun '{KAGGLE_DIR}'")   # kalau tak ketemu, beri pesan jelas

df_train = pd.read_csv(find_file("train.csv"))         # baca data latih → DataFrame
df_test = pd.read_csv(find_file("test.csv"))           # baca data uji (tanpa kolom target)

print('Ukuran data latih = {}'.format(df_train.shape))                                   # (jumlah baris, jumlah kolom) train
print('Memori data latih = {:.2f} MB'.format(df_train.memory_usage().sum() / 1024**2))   # memori: byte → MB
print('Ukuran data uji = {}'.format(df_test.shape))                                        # (jumlah baris, jumlah kolom) test
print('Memori data uji = {:.2f} MB'.format(df_test.memory_usage().sum() / 1024**2))        # memori test dalam MB

In [ ]:
df_train.head()   # tampilkan 5 baris pertama data train (kolom: id, keyword, location, text, target)

In [ ]:
df_test.head()    # tampilkan 5 baris pertama data test (perhatikan: tidak ada kolom target)

# 3. Eksplorasi data

Kita cek **panjang teks** (dalam karakter) untuk memastikan `MAX_LEN` aman. Tweet dibatasi ±280 karakter, dan satu token rata-rata ±3–4 karakter, jadi 160 token sudah lebih dari cukup.

Kita juga lihat **proporsi kelas**. Kalau sangat timpang, akurasi bisa menyesatkan — itu sebabnya kompetisi ini memakai F1.

> 💡 Ingin EDA lebih dalam? Lihat [notebook ini](https://www.kaggle.com/code/gunesevitan/nlp-with-disaster-tweets-eda-cleaning-and-bert). Semua tweet berbahasa Inggris.

In [ ]:
df_train["panjang"] = df_train["text"].apply(lambda x : len(x))   # kolom baru: jumlah karakter tiap tweet (train)
df_test["panjang"] = df_test["text"].apply(lambda x : len(x))     # idem untuk test

print("Statistik panjang tweet (data latih)")                  # judul
print(df_train["panjang"].describe())        # ringkasan statistik: count, mean, std, min, kuartil, max
print()                                     # baris kosong pemisah

print("Statistik panjang tweet (data uji)")                   # judul
print(df_test["panjang"].describe())         # ringkasan statistik panjang tweet test
print()                                     # baris kosong

print("Proporsi kelas pada data latih:")                          # judul
print(df_train["target"].value_counts(normalize=True).round(3))   # persentase tiap kelas (0 / 1)

# 4. Membagi data latih / validasi

Kita sisihkan **20% data** sebagai *validation set*: data ini **tidak dipakai untuk melatih**, hanya untuk mengukur seberapa baik model pada data baru.

```text
 df_train (7613 tweet)
 ├── 80%  X_train, y_train → dipakai untuk update bobot
 └── 20%  X_val,   y_val   → dipakai untuk mengukur performa tiap epoch
 df_test (3263 tweet, tanpa label) → hanya untuk submission
```

`to_clean_list` memastikan setiap tweet adalah `str` murni (tidak ada `NaN`), karena tokenizer akan error kalau menerima `NaN`.

In [ ]:
def to_clean_list(series):                                  # fungsi: pandas Series → list of str yang bersih
    """Isi NaN dengan string kosong, pastikan semuanya str, lalu jadikan list."""
    return series.fillna("").astype(str).tolist()           # NaN→"" ; paksa tipe str ; ubah ke list Python

X = to_clean_list(df_train["text"])                         # seluruh teks tweet train (fitur)
y = df_train["target"].to_numpy()                           # seluruh label train sebagai array NumPy (0/1)

X_train, X_val, y_train, y_val = train_test_split(          # bagi data secara acak
    X, y,                                                   #   data yang dibagi: teks & label (dipotong dengan indeks yang sama)
    test_size=VAL_SPLIT,                                    #   20% jadi validation
    random_state=SEED                                       #   seed agar pembagian selalu sama
)
X_test = to_clean_list(df_test["text"])                     # teks tweet test (tidak ada label)

print(f"Latih: {len(X_train)} | Validasi: {len(X_val)} | Uji: {len(X_test)}")   # cek jumlah tiap bagian
print(X_train[:3])                                          # intip 3 tweet pertama

# 5. Memuat model & tokenizer

## 5.1 Apa itu DistilBERT?
BERT adalah Transformer **encoder** yang membaca kalimat dua arah. **DistilBERT** adalah versi ringkasnya:

| | BERT-base | DistilBERT |
|---|---|---|
| Jumlah layer | 12 | **6** |
| Parameter | ±110 jt | **±66 jt** |
| Kecepatan | 1× | **±1.6×** |
| Performa (GLUE) | 100% | **±95–97%** |

## 5.2 Dua komponen yang dimuat
1. **Tokenizer** — mengubah teks menjadi angka (ID token).
2. **Model** — DistilBERT + *classification head* (lapisan linear kecil di atas) yang menghasilkan **2 logit**: skor kelas 0 dan kelas 1.

> ⚠️ **Peringatan "Some weights … newly initialized" itu normal.** Bobot DistilBERT dimuat dari hasil pra-latih, sedangkan classification head baru dibuat secara acak — dan memang itulah bagian yang akan dilatih.

> ⚠️ **Format folder lokal.** `from_pretrained` untuk PyTorch butuh `config.json` + `model.safetensors`/`pytorch_model.bin`. Preset KerasNLP (`model.weights.h5`) **tidak** bisa dibaca; fungsi `is_hf_folder` mengecek ini lebih dulu.

In [ ]:
def is_hf_folder(folder):                                   # fungsi: cek apakah folder berisi model format Hugging Face
    """True jika folder berisi model format Hugging Face (bukan preset Keras)."""
    if not os.path.isdir(folder):                           # folder tidak ada?
        return False                                        #   → jelas bukan model valid
    has_config = os.path.exists(os.path.join(folder, "config.json"))   # ada file konfigurasi arsitektur?
    has_weights = any(                                      # ada salah satu file bobot?
        os.path.exists(os.path.join(folder, f))             #   cek keberadaan tiap kandidat nama file
        for f in ["model.safetensors", "pytorch_model.bin"] #   dua format bobot yang didukung
    )
    return has_config and has_weights                       # valid hanya jika dua-duanya ada

if is_hf_folder(MODEL_DIR):                                 # folder lokal valid?
    model_source = MODEL_DIR                                #   → pakai folder lokal (tanpa internet)
    print(f"Memuat model dari folder lokal: {MODEL_DIR}")   #   info ke pengguna
else:                                                       # folder lokal tidak valid
    model_source = FALLBACK_MODEL                           #   → pakai nama model di Hugging Face Hub
    print(f"'{MODEL_DIR}' bukan folder model Hugging Face yang valid "
          f"(butuh config.json + model.safetensors/pytorch_model.bin).")   # jelaskan penyebabnya
    if os.path.isdir(MODEL_DIR):                            #   folder ada tapi formatnya salah?
        print("Isi folder:", os.listdir(MODEL_DIR))         #   tampilkan isinya untuk membantu debugging
    print(f"Fallback: mengunduh '{FALLBACK_MODEL}' dari Hugging Face Hub (butuh internet).")   # beri tahu akan unduh

tokenizer = AutoTokenizer.from_pretrained(model_source)     # muat tokenizer (vocab.txt, aturan pemotongan kata)
model = AutoModelForSequenceClassification.from_pretrained( # muat DistilBERT + head klasifikasi
    model_source,                                           #   sumber bobot (folder lokal atau Hub)
    num_labels=2                                            #   head menghasilkan 2 logit (kelas 0 dan 1)
)
model.to(device)                                            # pindahkan semua bobot model ke GPU/CPU yang dipilih

n_params = sum(p.numel() for p in model.parameters())       # jumlah seluruh parameter = Σ jumlah elemen tiap tensor bobot
print(f"\nModel siap. Jumlah parameter: {n_params/1e6:.1f} M")   # tampilkan dalam juta

## 5.3 Mengintip tokenisasi

Sebelum masuk ke model, tweet dipecah menjadi **subword token** (WordPiece), lalu tiap token diganti ID-nya. Dua token khusus ditambahkan otomatis:

| Token | Fungsi |
|---|---|
| `[CLS]` | Di awal kalimat; representasinya dipakai untuk klasifikasi |
| `[SEP]` | Di akhir kalimat sebagai penanda batas |

Kata yang tidak dikenal dipecah menjadi potongan (`##…`), misalnya `earthquake` → `earth`, `##quake`.

In [ ]:
contoh = X_train[0]                                         # ambil satu tweet sebagai contoh
print("Teks      :", contoh)                                 # tampilkan teks asli
print("Token     :", tokenizer.tokenize(contoh))             # hasil pemecahan jadi subword (belum ada token khusus)
enc = tokenizer(contoh, truncation=True, max_length=MAX_LEN) # tokenisasi lengkap: tambah [CLS]/[SEP], potong jika > MAX_LEN
print("input_ids :", enc["input_ids"])                       # ID angka tiap token (inilah yang masuk ke model)
print("Kembali   :", tokenizer.decode(enc["input_ids"]))     # decode balik ke teks untuk memastikan [CLS]/[SEP] ada
print("Keys      :", list(enc.keys()))                       # input_ids + attention_mask (1 = token asli, 0 = padding)

# 6. DataLoader

Model tidak menerima satu tweet-satu tweet, melainkan **batch**. Di PyTorch ada dua komponen:

| Komponen | Tugas |
|---|---|
| `Dataset` | Mendefinisikan **cara mengambil 1 contoh** (`__getitem__`) dan **jumlah total** (`__len__`) |
| `DataLoader` | Mengambil banyak contoh, mengacak urutan (opsional), lalu menyusunnya menjadi batch lewat `collate_fn` |

## Padding dinamis (*dynamic padding*)
Tweet panjangnya berbeda-beda, sedangkan tensor harus persegi. Daripada memaksa semuanya ke 160 token, `collate_fn` mem-*pad* hanya sampai **tweet terpanjang di batch itu**.

```text
Batch berisi 3 tweet:          input_ids setelah padding (0 = [PAD])
  "Fire in the forest"   →   [101, 2543, 1999, 1996, 3224, 102,    0,    0]
  "Lol so funny"         →   [101, 4879, 2061, 6057,  102,   0,    0,    0]
  "Earthquake hits city, ..."→[101, 8372, 4978, 2103, 1010, ..., 102]
attention_mask: 1 untuk token asli, 0 untuk padding → model mengabaikan padding
```

> 🧠 Bentuk tensor yang dihasilkan: `input_ids` dan `attention_mask` berukuran **(B, L)** → *B* = jumlah tweet di batch, *L* = panjang token terpanjang di batch.

In [ ]:
class TweetDataset(Dataset):                                # kelas Dataset kustom
    def __init__(self, texts, labels=None):                 # konstruktor: simpan data mentah
        self.texts = texts                                  #   list teks tweet
        self.labels = labels                                #   array label (None untuk data test)

    def __len__(self):                                      # dipanggil DataLoader untuk tahu ukuran data
        return len(self.texts)                              #   jumlah tweet

    def __getitem__(self, idx):                             # dipanggil DataLoader untuk mengambil contoh ke-idx
        item = {"text": self.texts[idx]}                    #   bungkus teks ke dictionary
        if self.labels is not None:                         #   kalau ada label (train/val)
            item["label"] = int(self.labels[idx])           #   tambahkan label sebagai int Python
        return item                                         #   satu contoh mentah (belum ditokenisasi)


def collate_fn(batch):                                      # menggabungkan list contoh → 1 batch tensor
    enc = tokenizer(                                        #   tokenisasi seluruh teks di batch sekaligus
        [b["text"] for b in batch],                         #     kumpulkan teks tiap contoh
        padding=True,                                       #     pad sampai panjang terpanjang di batch (dynamic padding)
        truncation=True,                                    #     potong jika melebihi max_length
        max_length=MAX_LEN,                                 #     batas atas panjang token
        return_tensors="pt",                                #     keluaran berupa tensor PyTorch
    )
    if "label" in batch[0]:                                 #   kalau data berlabel
        enc["labels"] = torch.tensor(                       #     buat tensor label
            [b["label"] for b in batch], dtype=torch.long   #       kumpulkan label, tipe integer 64-bit (syarat loss klasifikasi)
        )                                                   #     nama "labels" membuat model otomatis menghitung loss
    return enc                                              #   kembalikan dict tensor: input_ids, attention_mask, labels


def make_loader(texts, labels=None, shuffle=False, batch_size=BATCH_SIZE):   # pabrik DataLoader agar ringkas
    return DataLoader(                                      #   buat DataLoader
        TweetDataset(texts, labels),                        #     sumber data
        batch_size=batch_size,                              #     jumlah contoh per batch
        shuffle=shuffle,                                    #     True → urutan diacak tiap epoch (untuk train)
        collate_fn=collate_fn,                              #     fungsi penyusun batch di atas
    )


train_loader = make_loader(X_train, y_train, shuffle=True)                              # loader train: diacak
val_loader = make_loader(X_val, y_val, shuffle=False, batch_size=BATCH_SIZE * 2)        # loader val: tak diacak, batch lebih besar (tanpa backward hemat memori)

sample = next(iter(train_loader))                           # ambil 1 batch contoh untuk dicek
print({k: tuple(v.shape) for k, v in sample.items()})       # cetak bentuk tiap tensor, mis. input_ids: (32, 51)

# 7. Pelatihan model

## 7.1 Satu langkah pelatihan (satu batch)

```text
1. Forward   : batch → DistilBERT → logits (B, 2)
2. Loss      : bandingkan logits dengan label asli (cross-entropy)
3. Backward  : hitung gradien dL/dθ untuk semua bobot
4. Clip      : potong gradien yang terlalu besar (stabilitas)
5. Step      : optimizer memperbarui bobot
6. Scheduler : ubah learning rate sesuai jadwal
```

## 7.2 Rumus penting

**Cross-entropy loss** (kecil = prediksi benar & yakin):

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^{B}\log p_\theta(y_i \mid x_i), \qquad p = \mathrm{softmax}(\text{logits})$$

**Jadwal learning rate** (warmup linear lalu turun linear sampai 0), dengan $T$ = total langkah, $T_w$ = langkah warmup:

$$\eta_t=\begin{cases}\eta_{\max}\cdot \dfrac{t}{T_w} & t < T_w\\[2mm] \eta_{\max}\cdot\dfrac{T-t}{T-T_w} & t \ge T_w\end{cases}$$

**AdamW** = Adam + peluruhan bobot (*weight decay*) yang dipisah dari gradien — pilihan standar untuk fine-tuning Transformer.

<details>
<summary><b>▸ Detail: kenapa ada <code>GradScaler</code> dan <code>unscale_</code>?</b></summary>

Di float16, gradien yang sangat kecil bisa menjadi 0 (*underflow*). `GradScaler` mengalikan loss dengan faktor besar sebelum `backward()` agar gradien tetap terbaca. Sebelum *gradient clipping*, gradien harus dikembalikan ke skala asli lewat `unscale_`, kalau tidak ambang clipping jadi tidak bermakna. Jika AMP mati (`enabled=False`), `GradScaler` otomatis tidak melakukan apa-apa.
</details>

> 💡 **Kenapa `optimizer.zero_grad()` tiap batch?** PyTorch **menjumlahkan** gradien dari `backward()` sebelumnya. Tanpa di-nol-kan, gradien batch lama ikut tercampur.

In [ ]:
optimizer = torch.optim.AdamW(                              # optimizer AdamW
    model.parameters(),                                     #   seluruh bobot yang akan diupdate
    lr=LR,                                                  #   learning rate puncak
    weight_decay=WEIGHT_DECAY                               #   regularisasi L2 terpisah
)

total_steps = len(train_loader) * EPOCHS                    # total langkah = (jumlah batch per epoch) × (jumlah epoch)
scheduler = get_linear_schedule_with_warmup(                # penjadwal LR (rumus pada markdown di atas)
    optimizer,                                              #   optimizer yang LR-nya diatur
    num_warmup_steps=int(WARMUP_RATIO * total_steps),       #   jumlah langkah warmup (10% dari total)
    num_training_steps=total_steps,                         #   total langkah training
)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)      # skala loss untuk fp16; otomatis nonaktif jika bukan GPU NVIDIA


def autocast_ctx():                                         # fungsi pembuat konteks "presisi campuran"
    if use_amp:                                             #   AMP aktif (GPU NVIDIA)?
        return torch.autocast(device_type="cuda", dtype=torch.float16)   #   hitungan forward memakai float16
    return contextlib.nullcontext()                         #   kalau tidak, konteks kosong (presisi normal)


def to_device(batch):                                       # pindahkan semua tensor dalam batch ke device
    return {k: v.to(device) for k, v in batch.items()}      #   dict comprehension: tiap tensor .to(device)

In [ ]:
def train_one_epoch(epoch):                                 # fungsi: latih model selama satu epoch penuh
    model.train()                                           #   mode train → dropout aktif
    running_loss, correct, seen = 0.0, 0, 0                 #   akumulator: total loss, jumlah benar, jumlah contoh
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [latih]")   #   bungkus loader dengan bilah kemajuan
    for batch in pbar:                                      #   iterasi tiap batch
        batch = to_device(batch)                            #     kirim batch ke GPU/CPU
        optimizer.zero_grad(set_to_none=True)               #     kosongkan gradien lama (set_to_none lebih hemat memori)

        with autocast_ctx():                                #     forward pass dalam presisi campuran (jika aktif)
            out = model(**batch)                            #       ** membongkar dict jadi argumen: input_ids, attention_mask, labels
                                                            #       karena ada 'labels', out.loss dan out.logits terisi

        scaler.scale(out.loss).backward()                   #     backward: hitung gradien dari loss (yang diskalakan)
        scaler.unscale_(optimizer)                          #     kembalikan gradien ke skala asli sebelum di-clip
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   #     potong norma gradien maksimal 1.0 (cegah "meledak")
        scaler.step(optimizer)                              #     update bobot (dilewati otomatis jika gradien inf/NaN)
        scaler.update()                                     #     sesuaikan faktor skala untuk langkah berikutnya
        scheduler.step()                                    #     geser learning rate ke nilai langkah berikutnya

        bs = batch["labels"].size(0)                        #     jumlah contoh di batch ini
        running_loss += out.loss.item() * bs                #     tambah loss total (loss rata-rata × jumlah contoh)
        correct += (out.logits.argmax(dim=-1) == batch["labels"]).sum().item()   #     hitung prediksi benar (kelas dengan logit terbesar)
        seen += bs                                          #     total contoh yang sudah diproses
        pbar.set_postfix(loss=running_loss / seen, acc=correct / seen)   #     tampilkan loss & akurasi berjalan di bilah kemajuan
    return running_loss / seen, correct / seen              #   kembalikan rata-rata loss & akurasi epoch ini


@torch.no_grad()                                            # dekorator: matikan penghitungan gradien (hemat memori & cepat)
def evaluate(loader):                                       # fungsi: ukur performa pada satu loader berlabel
    model.eval()                                            #   mode eval → dropout dimatikan agar hasil deterministik
    running_loss, seen = 0.0, 0                             #   akumulator loss & jumlah contoh
    all_preds, all_labels = [], []                          #   tempat mengumpulkan prediksi & label asli
    for batch in tqdm(loader, desc="Evaluasi", leave=False):    # iterasi tiap batch (bilah kemajuan hilang setelah selesai)
        batch = to_device(batch)                            #     kirim ke device
        with autocast_ctx():                                #     presisi campuran bila aktif
            out = model(**batch)                            #     forward saja (tanpa backward)
        bs = batch["labels"].size(0)                        #     jumlah contoh di batch
        running_loss += out.loss.item() * bs                #     akumulasi loss
        seen += bs                                          #     akumulasi jumlah contoh
        all_preds.append(out.logits.argmax(dim=-1).cpu())   #     simpan kelas prediksi (pindah ke CPU)
        all_labels.append(batch["labels"].cpu())            #     simpan label asli
    preds = torch.cat(all_preds).numpy()                    #   gabung semua batch → 1 array prediksi
    labels = torch.cat(all_labels).numpy()                  #   gabung semua label asli
    return running_loss / seen, (preds == labels).mean(), f1_score(labels, preds)   # loss, akurasi, F1 (kelas 1)

## 7.3 Menjalankan pelatihan

Setiap epoch: **latih → validasi → simpan bobot jika F1 validasi membaik**. Setelah semua epoch, bobot terbaik dikembalikan.

> 💡 **Cara membaca log:** jika `loss latih` terus turun tetapi `loss val` mulai naik, itu tanda *overfitting* — model mulai menghafal data latih.

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}   # catatan metrik per epoch
best_f1, best_state = -1.0, None                            # F1 terbaik sejauh ini & salinan bobotnya

for epoch in range(1, EPOCHS + 1):                          # ulangi untuk epoch 1..EPOCHS
    tr_loss, tr_acc = train_one_epoch(epoch)                #   latih satu epoch
    val_loss, val_acc, val_f1 = evaluate(val_loader)        #   ukur performa di data validasi

    history["train_loss"].append(tr_loss)                   #   catat loss train
    history["train_acc"].append(tr_acc)                     #   catat akurasi train
    history["val_loss"].append(val_loss)                    #   catat loss validasi
    history["val_acc"].append(val_acc)                      #   catat akurasi validasi
    history["val_f1"].append(val_f1)                        #   catat F1 validasi

    print(f"Epoch {epoch}/{EPOCHS} | loss latih {tr_loss:.4f} akurasi {tr_acc:.4f} | "
          f"loss val {val_loss:.4f} akurasi val {val_acc:.4f} F1 val {val_f1:.4f}")   # ringkasan satu baris per epoch

    if val_f1 > best_f1:                                    #   F1 validasi lebih baik dari sebelumnya?
        best_f1 = val_f1                                    #     perbarui rekor
        best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})   # salin bobot ke CPU (deepcopy agar tak ikut berubah)

# Kembalikan bobot terbaik (berdasarkan F1 validasi)
model.load_state_dict(best_state)                           # isi ulang model dengan bobot terbaik
model.to(device)                                            # pastikan model kembali berada di device
print(f"\nBobot terbaik dimuat (F1 validasi = {best_f1:.4f})")   # konfirmasi

# 8. Evaluasi

## 8.1 Metrik: F1-score

Dari matriks konfusi (*confusion matrix*): **TP** (bencana, ditebak bencana), **FP** (bukan bencana, ditebak bencana), **FN** (bencana, ditebak bukan).

$$P=\frac{TP}{TP+FP},\qquad R=\frac{TP}{TP+FN},\qquad F_1=\frac{2PR}{P+R}=\frac{2\,TP}{2\,TP+FP+FN}$$

|  | Prediksi: Bukan Bencana | Prediksi: Bencana |
|---|---|---|
| **Asli: Bukan Bencana** | TN ✅ | FP ❌ |
| **Asli: Bencana** | FN ❌ | TP ✅ |

> 🧠 F1 adalah rata-rata harmonik *precision* (ketepatan) dan *recall* (kelengkapan): nilainya tinggi hanya jika **keduanya** tinggi.

## 8.2 Dari logit ke prediksi
`np.argmax(logits, axis=1)` memilih kelas dengan skor tertinggi. Secara ekuivalen: prediksi `1` jika $p(\text{bencana}) > 0.5$.

In [ ]:
@torch.no_grad()                                            # tanpa gradien (inferensi)
def predict_logits(texts):                                  # fungsi: teks mentah → logits (N, 2)
    """Mengembalikan logits (N, 2) sebagai numpy array."""
    model.eval()                                            #   mode eval (dropout mati)
    loader = make_loader(texts, shuffle=False, batch_size=BATCH_SIZE * 2)   #   loader tanpa label & tanpa shuffle (urutan harus tetap!)
    outputs = []                                            #   wadah hasil tiap batch
    for batch in tqdm(loader, desc="Prediksi", leave=False): #   iterasi batch
        batch = to_device(batch)                            #     kirim ke device
        with autocast_ctx():                                #     presisi campuran bila aktif
            logits = model(**batch).logits                  #     forward; ambil logits (B, 2)
        outputs.append(logits.float().cpu())                #     ubah ke float32 & pindah ke CPU
    return torch.cat(outputs).numpy()                       #   gabung semua batch → array (N, 2)


def displayConfusionMatrix(y_true, y_pred, dataset):        # fungsi: gambar confusion matrix + skor F1
    y_hat = np.argmax(y_pred, axis=1)                       #   logit → kelas prediksi (0/1)
    disp = ConfusionMatrixDisplay.from_predictions(         #   buat plot confusion matrix
        y_true,                                             #     label asli
        y_hat,                                              #     label prediksi
        display_labels=["Bukan Bencana", "Bencana"],        #     nama sumbu
        cmap=plt.cm.Blues                                   #     skema warna biru
    )
    f1 = f1_score(y_true, y_hat)                            #   hitung F1 kelas 1 memakai sklearn
    disp.ax_.set_title("Matriks Konfusi pada Data " + dataset + " -- Skor F1: " + str(round(f1, 2)))   # judul berisi nama dataset & F1

In [ ]:
y_pred_train = predict_logits(X_train)                      # logits untuk seluruh data train
displayConfusionMatrix(y_train, y_pred_train, "Latih")   # gambar confusion matrix train

In [ ]:
y_pred_val = predict_logits(X_val)                          # logits untuk data validasi
displayConfusionMatrix(y_val, y_pred_val, "Validasi")     # gambar confusion matrix validasi

> 🧠 **Cara membaca:** bandingkan F1 data *latih* dengan data *validasi*. Selisih besar (mis. 0.95 vs 0.82) menandakan overfitting; kedua angka rendah menandakan model kurang belajar (*underfitting*).

# 9. Membuat file submission

Untuk setiap tweet di test set kita prediksi `1` (bencana nyata) atau `0` (bukan). Format file yang diminta Kaggle:

```text
id,target
0,1
2,0
...
```

> ⚠️ **Urutan baris harus sama dengan `df_test`.** Itu sebabnya `shuffle=False` pada loader prediksi, dan `id` diambil langsung dari `df_test["id"]`.

In [ ]:
test_logits = predict_logits(X_test)                        # logits untuk seluruh tweet test, bentuk (3263, 2)

# Urutan baris = urutan df_test, jadi cukup ambil kolom id dari df_test
submission = pd.DataFrame({                                 # susun tabel submission
    "id": df_test["id"],                                    #   kolom id dari data test
    "target": np.argmax(test_logits, axis=1),               #   kolom target = kelas dengan logit terbesar
})
submission.head()                                           # intip 5 baris pertama

In [ ]:
submission["target"].value_counts()                         # sebaran prediksi (harusnya mirip ±43% kelas 1)

In [ ]:
submission.to_csv("submission.csv", index=False)            # simpan ke CSV tanpa kolom indeks pandas
print("submission.csv tersimpan:", submission.shape)        # konfirmasi ukuran file

## (Opsional) Simpan model hasil fine-tuning

Disimpan dalam format Hugging Face sehingga bisa dimuat ulang dengan `AutoModelForSequenceClassification.from_pretrained("distilbert_disaster_tweets_pt")`.

In [ ]:
SAVE_DIR = "distilbert_disaster_tweets_pt"                  # nama folder tujuan
model.save_pretrained(SAVE_DIR)                             # simpan bobot + config model
tokenizer.save_pretrained(SAVE_DIR)                         # simpan tokenizer (vocab, dsb.)
print("Model disimpan di:", SAVE_DIR)                       # konfirmasi lokasi

# 10. Rangkuman & keterbatasan baseline

**Hasil Kaggle: `0.83389`** (public leaderboard, F1).

### ✅ Yang sudah baik
- Pipeline PyTorch lengkap dan bisa direproduksi.
- Dynamic padding, AMP, gradient clipping, scheduler warmup.
- Bobot terbaik dipilih berdasarkan F1 validasi, bukan epoch terakhir.

### ⚠️ Keterbatasan yang menjadi target notebook 2

| # | Keterbatasan baseline | Dampak |
|---|---|---|
| 1 | Hanya **1 split 80/20** → hasil bergantung pada keberuntungan split | Estimasi skor berisik; hanya 80% data dipakai belajar |
| 2 | **Label tidak konsisten**: tweet yang sama muncul beberapa kali dengan label berbeda | Model dilatih dengan sinyal yang saling bertentangan |
| 3 | Kolom **`keyword` diabaikan** | Kehilangan petunjuk yang berguna |
| 4 | Teks mentah (URL, `@user`, `&amp;` apa adanya) | Token tidak informatif memenuhi input |
| 5 | Model kecil (**DistilBERT**) | Kapasitas representasi terbatas |
| 6 | **Satu model** saja | Variansi tinggi; tidak ada efek ensemble |
| 7 | Ambang keputusan **tetap 0.5** | Belum tentu optimal untuk F1 |
| 8 | Loss biasa pada label yang berisik | Model terlalu yakin pada label yang mungkin salah |

➡️ Lanjut ke **`disaster-tweets-roberta-cv-pytorch-id.ipynb`**.